In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
from google.colab import files
uploaded = files.upload()

Saving energydata_complete.xlsx to energydata_complete.xlsx


In [3]:
df = pd.read_excel("energydata_complete.xlsx")

df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [4]:
df.isna().sum()

,0
date,0
Appliances,0
lights,0
T1,0
RH_1,0
T2,0
RH_2,0
T3,0
RH_3,0
T4,0


In [ ]:
df.columns

Index(['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3',
       'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8',
       'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed',
       'Visibility', 'Tdewpoint', 'rv1', 'rv2'],
      dtype='object')

In [ ]:
# Baseline: testing linear relationship between temperature sensors T2 and T6
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = df[['T2']]
y = df['T6']
lr = LinearRegression()
lr.fit(X, y)
predictions = lr.predict(X)
rmse_T2_T6 = np.sqrt(mean_squared_error(y, predictions))
print(round(rmse_T2_T6, 3))

3.644


In [10]:
from sklearn.model_selection import train_test_split

df_clean = df.drop(['date', 'lights'], axis = 1)
target = df_clean['Appliances']
features = df_clean.drop('Appliances', axis = 1)
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.3, random_state=42)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Linear Regression — training set evaluation (MAE and RMSE)
from sklearn.metrics import mean_squared_error, mean_absolute_error

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
train_preds = lr.predict(X_train_scaled)
mae_train = mean_absolute_error(y_train, train_preds)
rmse_train = np.sqrt(mean_squared_error(y_train, train_preds))
print("MAE (Train):", round(mae_train, 3))
print("RMSE (Train):", round(rmse_train, 3))


MAE (Train): 53.742
RMSE (Train): 95.216


In [ ]:
# Linear Regression — test set evaluation
test_preds = lr.predict(X_test_scaled)
mae_test = mean_absolute_error(y_test, test_preds)
rmse_test = np.sqrt(mean_squared_error(y_test, test_preds))
print("MAE (Test):", round(mae_test, 3))
print("RMSE (Test):", round(rmse_test, 3))

MAE (Test): 53.643
RMSE (Test): 93.64


In [ ]:
# Overfitting check: comparing train vs test R²
from sklearn.metrics import r2_score

train_r2 = r2_score(y_train, train_preds)
test_r2 = r2_score(y_test, test_preds)
print("Train R² Score:", round(train_r2, 3))
print("Test R² Score:", round(test_r2, 3))

Train R² Score: 0.145
Test R² Score: 0.149


In [ ]:
# Ridge Regression — L2 regularisation to reduce overfitting
from sklearn.linear_model import Ridge

ridge_model = Ridge()
ridge_model.fit(X_train_scaled, y_train)
ridge_test_preds = ridge_model.predict(X_test_scaled)
ridge_rmse_test = np.sqrt(mean_squared_error(y_test, ridge_test_preds))
print(ridge_rmse_test)

93.70877477642011


In [ ]:
# Lasso Regression — L1 regularisation; checking how many features survive
from sklearn.linear_model import Lasso

lasso_model = Lasso()
lasso_model.fit(X_train_scaled, y_train)
non_zero_features = np.sum(lasso_model.coef_ != 0)
print(non_zero_features)

4


In [ ]:
# Lasso test set evaluation
lasso_test_preds = lasso_model.predict(X_test_scaled)
lasso_rmse_test = np.sqrt(mean_squared_error(y_test, lasso_test_preds))
print("Lasso RMSE (Test):", round(lasso_rmse_test, 3))

Lasso RMSE (Test): 99.424
